# Stock-Specific Models Training
This notebook is the Jupyter version of `train_models.py`. It trains separate models for each stock.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

### 1. Setup Root Path & Load Data

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

stock_files = ["reliance_feat.csv", "tcs_feat.csv", "infosys_feat.csv", "hdfcbank_feat.csv"]
stock_data_dict = {}

for file in stock_files:
    stock_name = file.split('_')[0].upper()
    if stock_name == 'INFOSYS': stock_name = 'INFY'
    temp_df = pd.read_csv(ROOT / "data" / "features" / file, index_col="Date", parse_dates=True)
    stock_data_dict[stock_name] = temp_df.sort_index()
    print(f"{stock_name} data loaded. Shape: {temp_df.shape}")

### 2. Define Features

In [ ]:
drop_cols = ["Close", "Ticker", "Adj Close", "Datetime", "Date"]
sample_df = stock_data_dict['RELIANCE']
feature_cols = [c for c in sample_df.columns if c not in drop_cols and not c.startswith("Target_")]
print("Number of training features:", len(feature_cols))

### 3. Training Loop (Per Stock, Per Horizon)

In [ ]:
horizons = [1, 7, 15]

for stock_name, df in stock_data_dict.items():
    print("="*50)
    print(f"Training Models for {stock_name}...")
    print("="*50)
    
    best_models_dict = {}
    main_scaler = None
    
    for n in horizons:
        print(f"\n  --- Training for {n}-Day Horizon ---")
        
        df[f"Target_{n}"] = (df["Close"].shift(-n) - df["Close"]) / df["Close"]
        valid_data = df.dropna(subset=[f"Target_{n}"])
        
        X = valid_data[feature_cols]
        y = valid_data[f"Target_{n}"]
        prices = valid_data["Close"]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)
        _, _, _, prices_test = train_test_split(X, prices, shuffle=False, test_size=0.2)
        
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
        
        models = {
            "LinearRegression": LinearRegression(),
            "RandomForest": RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_split=5, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "XGBoost": xgb.XGBRegressor(objective="reg:pseudohubererror", n_estimators=150, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42)
        }
        
        best_rmse = float('inf')
        best_model_name = None
        best_model_obj = None
        
        for name, model in models.items():
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
            
            curr_price = prices_test.values
            preds_price = curr_price * (1 + preds)
            actual_price = curr_price * (1 + y_test.values)
            rmse = np.sqrt(mean_squared_error(actual_price, preds_price))
            
            if rmse < best_rmse:
                best_rmse = rmse
                best_model_name = name
                best_model_obj = model
                
        print(f"  Best Model: {best_model_name} (RMSE: {best_rmse:.2f})")
        best_models_dict[n] = best_model_obj
        if n == 1:
            main_scaler = scaler
            
    stock_dir = ROOT / "models" / stock_name
    os.makedirs(stock_dir, exist_ok=True)
    
    joblib.dump(best_models_dict, stock_dir / "models.pkl")
    joblib.dump(main_scaler, stock_dir / "scaler.pkl")
    joblib.dump(feature_cols, stock_dir / "features.pkl")
    print(f"-> Models & Scaler saved for {stock_name} in {stock_dir}\n")

print("All done!")